## Funktionsaufruf

Funktionsaufrufe (auch Tool-Aufrufe genannt ) bieten OpenAI-Modellen eine leistungsstarke und flexible Möglichkeit, mit externen Systemen zu interagieren und auf Daten ausserhalb ihrer Trainingsdaten zuzugreifen. 

Betrachten wir den gesamten Ablauf eines Toolaufrufs für eine get_horoscopeFunktion, die ein tägliches Horoskop für ein Sternzeichen abruft.

In [ ]:
%run ~/data/env.py

In [ ]:
from openai import OpenAI
import json

client = OpenAI(api_key=OPENAI_API_KEY, base_url=AI_BASE_URL)

# 1. Define a list of callable tools for the model
tools = [
    {
        "type": "function",
        "name": "get_horoscope",
        "description": "Get today's horoscope for an astrological sign.",
        "parameters": {
            "type": "object",
            "properties": {
                "sign": {
                    "type": "string",
                    "description": "An astrological sign like Taurus or Aquarius",
                },
            },
            "required": ["sign"],
        },
    },
]

def get_horoscope(sign):
    return f"{sign}: Next Tuesday you will befriend a baby otter."

# Create a running input list we will add to over time
input_list = [
    {"role": "user", "content": "What is my horoscope? I am an Aquarius."}
]

# 2. Prompt the model with tools defined
response = client.responses.create(
    model=AI_MODEL,
    tools=tools,
    input=input_list,
)

# Save function call outputs for subsequent requests
input_list += response.output

for item in response.output:
    if item.type == "function_call":
        if item.name == "get_horoscope":
            # 3. Execute the function logic for get_horoscope
            sign = json.loads(item.arguments)["sign"]
            horoscope = get_horoscope(sign)
            
            # 4. Provide function call results to the model
            input_list.append({
                "type": "function_call_output",
                "call_id": item.call_id,
                "output": horoscope,
            })

print("Final input:")
print(input_list)

response = client.responses.create(
    model=AI_MODEL,
    instructions="Respond only with a horoscope generated by a tool.",
    tools=tools,
    input=input_list,
)

# 5. The model should be able to give a response!
print("Final output:")
print(response.model_dump_json(indent=2))
print("\n" + response.output_text)

---
### Weitere Beispiele

* [OpenAI API - Function calling](https://developers.openai.com/api/docs/guides/function-calling)